# Hybrid networks with PyTorch

`polypus.torch.QuantumLayer` turns a native model into a PyTorch layer, a `torch.nn.Module`, so it can sit between classical layers and train with any `torch.optim` optimizer. Its forward pass is `qml.expval` and its backward pass uses `qml.jacobian`, the two primitives of [`07c`](07c_expval_and_jacobian.ipynb): PyTorch's automatic differentiation computes the classical layers' gradients, and Polypus computes the quantum layer's exactly.

PyTorch is not installed with Polypus. It comes as an optional extra:

```bash
pip install "polypus-quantum[torch]"
```

`import polypus` does not import PyTorch; only `polypus.torch` does.

## The quantum layer

The model has two qubits and a readout of two observables, `⟨Z₀⟩` and `⟨Z₁⟩`, so the layer takes two angles per sample and returns two numbers per sample. Its trainable parameters are the model's, in `params`, a `torch.nn.Parameter` drawn at random from PyTorch's generator, so `torch.manual_seed` fixes them. `exact=True` uses exact expectations, without shot noise:

In [ ]:
import torch
from polypus import qml
from polypus.torch import QuantumLayer

torch.manual_seed(3)

model = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).real_amplitudes(1),
    qml.Readout([qml.Z(0), qml.Z(1)]),
)
quantum = QuantumLayer(model, exact=True)
print(quantum)
print(quantum.params)

Called on a batch of samples, the layer returns one row per sample and one column per observable, a tensor PyTorch can differentiate: `backward` fills `quantum.params.grad` with the exact gradient, computed by Polypus with the parameter-shift rule. The gradient is then cleared, so the training below starts clean:

In [ ]:
angles = torch.tensor([[0.2, 0.1], [2.8, 3.0]])
output = quantum(angles)
print(output)

output.sum().backward()
print(quantum.params.grad)
quantum.params.grad = None

## A hybrid network

There are now eight samples with four features each, four per class. A `Linear` layer maps the four features onto the two angles the quantum layer takes, and a second `Linear` turns its two outputs into one logit, the score of class 1. The classical layers work in `float32`, PyTorch's default; the quantum layer converts the type on the way in and out:

In [ ]:
X = torch.tensor(
    [
        [0.9, 0.8, 0.1, 0.0],
        [1.0, 0.6, 0.2, 0.1],
        [0.8, 0.9, 0.0, 0.2],
        [0.7, 1.0, 0.1, 0.1],
        [0.1, 0.2, 0.9, 1.0],
        [0.0, 0.1, 1.0, 0.8],
        [0.2, 0.0, 0.8, 0.9],
        [0.1, 0.1, 0.7, 1.0],
    ]
)
Y = torch.tensor([[1.0], [1.0], [1.0], [1.0], [0.0], [0.0], [0.0], [0.0]])

net = torch.nn.Sequential(
    torch.nn.Linear(4, 2),
    quantum,
    torch.nn.Linear(quantum.num_observables, 1),
)
print(net)

The training is ordinary PyTorch code: a binary cross-entropy loss, the `torch.optim.Adam` optimizer over every parameter of the network, quantum ones included, and 30 steps. Each step runs one `qml.expval` call forward and one `qml.jacobian` call backward:

In [ ]:
import matplotlib.pyplot as plt

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.1)


def accuracy():
    with torch.no_grad():
        return ((net(X) > 0).float() == Y).float().mean().item()


print("accuracy before:", accuracy())
losses = []
for step in range(30):
    optimizer.zero_grad()
    loss = loss_fn(net(X), Y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print("accuracy after: ", accuracy())

plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("binary cross-entropy")
plt.show()

## Starting from a trained model: `from_trained`

`from_trained` builds the layer from a `TrainedModel`, the result of `fit` and `bind` or of `qml.load`, so a model trained with Polypus, as in [`07a`](07a_native_classifier.ipynb), can continue its training inside a PyTorch network. The new layer starts at the trained parameters, so its outputs match `qml.expval`'s for them:

In [ ]:
import numpy as np
import polypus
from polypus.torch import from_trained

x_small = np.array([[0.2, 0.1], [0.4, 0.3], [2.7, 2.9], [3.0, 2.6]])
classifier = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).real_amplitudes(1),
    qml.Readout([qml.Z(0)], decision="sign"),
)
result = classifier.fit(
    x_small,
    [1, 1, -1, -1],
    method=polypus.Adam(max_iters=30),
    loss="hinge",
    exact=True,
    seed=7,
)
trained = classifier.bind(result.best_params)

layer = from_trained(trained, exact=True)
with torch.no_grad():
    print(layer(torch.tensor(x_small)).squeeze(1))
print(qml.expval(classifier, x_small, trained.params, exact=True)[:, 0])

## Summary

This notebook turned a native model into a PyTorch layer with `QuantumLayer`, checked its outputs and its exact gradient, placed it between two `Linear` layers in a hybrid network trained with `torch.optim.Adam`, and, with `from_trained`, started a layer from a model trained with `fit`.

## End of the series

This series has walked the full path: from the first circuit to training models with data. Readers who want to go further can check [`examples/`](https://github.com/Bahia-Software/polypus/tree/main/examples), with bigger cases, including a full MaxCut with QAOA example. Several of them continue this section: `qml_iris_demo.py` trains a native classifier on the Iris flowers, and `qml_native_showcase.py` goes through every native encoder, layer, optimizer, readout and loss. `qml_expval_jacobian_loop.py` and `qml_torch_hybrid.py` are the scripts behind [`07c`](07c_expval_and_jacobian.ipynb) and this notebook. The [user guide](https://github.com/Bahia-Software/polypus/tree/main/docs/guide) covers training, QML, memory and performance in depth; its [QML chapter](https://github.com/Bahia-Software/polypus/blob/main/docs/guide/qml.md) documents `polypus.torch` in full.

Back to the index: [`00_welcome.ipynb`](00_welcome.ipynb).